# 📓 Exemplo 04: Balanço hídrico

**Missão 3** · [![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/change_climate/blob/main/code/notebooks_exemplo/04_balanco_hidrico.ipynb)

> **Pergunta:** em que meses sobra e em que meses falta água no solo?

### 🧭 Como usar este notebook
1. 💾 **Faça uma cópia:** menu **Arquivo → Salvar uma cópia no Drive**. Mova a cópia para a pasta
   `mudanca_climatica_br/notebooks/`. Assim você pode mudar tudo sem estragar o original.
2. ▶️ **Rode uma célula por vez** (botão ▶ ou **Shift + Enter**) e **leia os comentários** (as linhas com `#`).
3. 🧪 Os exemplos usam **Santa Helena-PR** e **Curitiba-PR**.
4. ✍️ No fim, a seção **Sua vez** pede para você **replicar o método para as capitais do Brasil**.

In [ ]:
# Instala as bibliotecas que não vêm no Colab (só precisa rodar uma vez por sessão)
!pip install agrometeorologiapy pymannkendall

In [ ]:
# Normal climatológica 1991-2020 (base do Xavier), janeiro a dezembro

# ---------- Santa Helena-PR ----------
latitude_sh = -24.86
temperatura_sh = [27.0, 27.0, 26.3, 23.9, 20.3, 19.3, 18.9, 20.8, 22.6, 24.6, 25.4, 26.5]   # °C
chuva_sh = [172.4, 157.2, 131.5, 140.0, 183.1, 140.4, 100.4, 84.1, 139.4, 227.8, 186.9, 186.6]   # mm

# ---------- Curitiba-PR ----------
latitude_ct = -25.43
temperatura_ct = [22.4, 22.5, 21.4, 19.6, 16.4, 15.3, 14.8, 16.0, 17.2, 18.8, 20.0, 21.7]   # °C
chuva_ct = [205.8, 177.0, 142.3, 80.5, 88.0, 109.3, 98.3, 82.5, 136.1, 160.7, 119.4, 152.1]   # mm

meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

## 1. Tabela do clima e ETP de Thornthwaite (Santa Helena)

In [ ]:
import agrometeorologiapy as amp
import pandas as pd
import matplotlib.pyplot as plt

# Uma tabela (DataFrame) é como uma planilha: cada coluna tem um nome
clima_sh = pd.DataFrame({
    'Meses': meses,
    'T_media_C': temperatura_sh,
    'P (mm/mês)': chuva_sh,
})

# ETP mensal de Thornthwaite: a função devolve a tabela com a coluna nova 'ETP_mm_mes'
clima_sh = amp.thornthwaite_mensal(clima_sh, lat=latitude_sh)
clima_sh

## 2. O balanço hídrico (CAD = 100 mm)

In [ ]:
# A função do balanço espera a coluna com este nome exato
clima_sh['ETP (mm/mês)'] = clima_sh['ETP_mm_mes']

balanco_sh = amp.balanco_hidrico_climatologico(clima_sh, CAD=100)

# Mostra só as colunas principais
balanco_sh[['Meses', 'P (mm/mês)', 'ETP (mm/mês)', 'ARM (mm/mês)', 'ETR (mm/mês)', 'DEF (mm/mês)', 'EXC (mm/mês)']].round(1)

In [ ]:
print('ETP anual (mm):', round(balanco_sh['ETP (mm/mês)'].sum()))
print('Deficiência anual (mm):', round(balanco_sh['DEF (mm/mês)'].sum()))
print('Excedente anual (mm):', round(balanco_sh['EXC (mm/mês)'].sum()))

## 3. A mesma coisa para Curitiba

In [ ]:
clima_ct = pd.DataFrame({'Meses': meses, 'T_media_C': temperatura_ct, 'P (mm/mês)': chuva_ct})
clima_ct = amp.thornthwaite_mensal(clima_ct, lat=latitude_ct)
clima_ct['ETP (mm/mês)'] = clima_ct['ETP_mm_mes']
balanco_ct = amp.balanco_hidrico_climatologico(clima_ct, CAD=100)

print('ETP anual (mm):', round(balanco_ct['ETP (mm/mês)'].sum()))
print('Deficiência anual (mm):', round(balanco_ct['DEF (mm/mês)'].sum()))
print('Excedente anual (mm):', round(balanco_ct['EXC (mm/mês)'].sum()))

## 4. Gráfico: excedente para cima, deficiência para baixo

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

eixos[0].bar(meses, balanco_sh['EXC (mm/mês)'], color='steelblue', label='Excedente')
eixos[0].bar(meses, -balanco_sh['DEF (mm/mês)'], color='orange', label='Deficiência')
eixos[0].axhline(0, color='black')
eixos[0].set_title('Santa Helena-PR')
eixos[0].set_ylabel('mm/mês')
eixos[0].legend()

eixos[1].bar(meses, balanco_ct['EXC (mm/mês)'], color='steelblue')
eixos[1].bar(meses, -balanco_ct['DEF (mm/mês)'], color='orange')
eixos[1].axhline(0, color='black')
eixos[1].set_title('Curitiba-PR')

plt.tight_layout()
plt.show()

## 5. Experimento: e se esquentar 2 °C?

In [ ]:
# Cria uma lista nova somando 2 °C a cada mês (um mês por vez, com for)
temperatura_mais_2 = []
for t in temperatura_sh:
    temperatura_mais_2.append(t + 2)

clima_teste = pd.DataFrame({'Meses': meses, 'T_media_C': temperatura_mais_2, 'P (mm/mês)': chuva_sh})
clima_teste = amp.thornthwaite_mensal(clima_teste, lat=latitude_sh)
clima_teste['ETP (mm/mês)'] = clima_teste['ETP_mm_mes']
balanco_teste = amp.balanco_hidrico_climatologico(clima_teste, CAD=100)

print('Deficiência anual atual (mm):', round(balanco_sh['DEF (mm/mês)'].sum()))
print('Deficiência anual com +2 °C (mm):', round(balanco_teste['DEF (mm/mês)'].sum()))

> 🧠 A chuva não mudou, mas a deficiência mudou. Por quê?

## ✍️ Sua vez
1. Faça o balanço hídrico de **5 capitais** (normais do INMET, com o modelo abaixo), incluindo uma do semiárido e uma da Amazônia.
2. Monte uma tabela: capital | ETP anual | DEF anual | EXC anual.
3. Repita o experimento de +2 °C para as 5. Em qual a deficiência aumenta mais?

In [ ]:
# ✍️ MODELO para uma capital. Copie esta célula uma vez para cada capital.
#
# Onde achar os números: portal do INMET, Normais Climatológicas 1991-2020
#   https://portal.inmet.gov.br/normais
# Procure a estação da capital e anote, de janeiro a dezembro:
#   - temperatura média compensada (°C)
#   - precipitação acumulada (mm)
# A latitude da estação aparece na mesma tabela (negativa no Hemisfério Sul).

capital = '___'                 # ex.: 'Brasília'
latitude_capital = ___          # ex.: -15.79
temperatura_capital = [___, ___, ___, ___, ___, ___, ___, ___, ___, ___, ___, ___]
chuva_capital = [___, ___, ___, ___, ___, ___, ___, ___, ___, ___, ___, ___]

# Depois, use estas listas no lugar de temperatura_sh, chuva_sh e latitude_sh nas células do exemplo.